What the script does

Walk all species folders under Fish_Dataset/Fish_Dataset/*/*.

Match each image with its mask in the * GT folder (same basename preferred).

Read mask → threshold to binary → findContours → keep the largest external contour (usually the fish).

Approximate the contour to a polygon (reduce points), normalize to [0..1] by width/height.

Write labels/<split>/<basename>.txt with one line: 0 x1 y1 x2 y2 ... (class 0 = fish).

Copy the image to images/<split>/.

If a mask is missing, the script will skip the image.

In [1]:
import os, cv2, glob, random, shutil
import numpy as np

# ---- configure these ----
ROOT = r"DATA/LARGE SCALE FISH DATASET/Fish_Dataset/Fish_Dataset"
OUT  = r"DATA/YOLO_FISH_SEG"
VAL_FRACTION = 0.15
RANDOM_SEED = 42
MIN_POLY_POINTS = 12      # increase for smoother polygons
APPROX_EPS_FRAC = 0.002   # polygon simplification; lower = more points
# -------------------------

random.seed(RANDOM_SEED)
os.makedirs(os.path.join(OUT, "images", "train"), exist_ok=True)
os.makedirs(os.path.join(OUT, "images", "val"),   exist_ok=True)
os.makedirs(os.path.join(OUT, "labels", "train"), exist_ok=True)
os.makedirs(os.path.join(OUT, "labels", "val"),   exist_ok=True)

# collect all species subfolders (each has "<Species>\ <Species> GT")
species_dirs = [d for d in glob.glob(os.path.join(ROOT, "*")) if os.path.isdir(d)]

pairs = []  # (img_path, mask_path)
for sp_dir in species_dirs:
    # find image folder and GT folder
    subs = [d for d in glob.glob(os.path.join(sp_dir, "*")) if os.path.isdir(d)]
    img_dir = [d for d in subs if os.path.basename(d).lower().endswith("gt")==False]
    gt_dir  = [d for d in subs if os.path.basename(d).lower().endswith("gt")==True]
    if not img_dir or not gt_dir:
        continue
    img_dir, gt_dir = img_dir[0], gt_dir[0]

    # index masks by basename (without extension)
    mask_index = {}
    for m in glob.glob(os.path.join(gt_dir, "*")):
        base = os.path.splitext(os.path.basename(m))[0]
        mask_index[base] = m

    # match images to masks
    for img in glob.glob(os.path.join(img_dir, "*")):
        base = os.path.splitext(os.path.basename(img))[0]
        # handle occasional naming differences (spaces, case)
        cand = [
            base,
            base.replace(" ", "_"),
            base.replace("_", " "),
            base.lower(), base.upper()
        ]
        mask = None
        for c in cand:
            if c in mask_index:
                mask = mask_index[c]
                break
        if mask is None:
            # fallback: try any mask that startswith the same prefix
            for k,v in mask_index.items():
                if k.startswith(base):
                    mask = v; break
        if mask:
            pairs.append((img, mask))

# split
random.shuffle(pairs)
n_val = int(len(pairs) * VAL_FRACTION)
val_pairs = set(pairs[:n_val])

def write_label(mask_path, out_txt):
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        return False
    # binarize
    _, bw = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
    # find contours
    contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return False
    # take largest contour (assume single fish)
    cnt = max(contours, key=cv2.contourArea)

    # approximate polygon
    eps = APPROX_EPS_FRAC * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, eps, True).reshape(-1, 2)

    # ensure minimum points (fallback to raw contour if too few)
    if len(approx) < MIN_POLY_POINTS:
        approx = cnt.reshape(-1, 2)

    h, w = bw.shape[:2]
    # normalize to [0,1]
    poly = []
    for x,y in approx:
        poly += [x / w, y / h]

    # YOLOv8-seg: class_id followed by x y pairs
    with open(out_txt, "w", encoding="utf-8") as f:
        line = "0 " + " ".join(f"{v:.6f}" for v in poly)
        f.write(line + "\n")
    return True

# convert & copy
for img, mask in pairs:
    split = "val" if (img, mask) in val_pairs else "train"
    out_img = os.path.join(OUT, "images", split, os.path.basename(img))
    out_txt = os.path.join(OUT, "labels", split, os.path.splitext(os.path.basename(img))[0] + ".txt")

    os.makedirs(os.path.dirname(out_img), exist_ok=True)
    os.makedirs(os.path.dirname(out_txt), exist_ok=True)

    ok = write_label(mask, out_txt)
    if ok:
        shutil.copy2(img, out_img)

print("Done. Created YOLO dataset at:", OUT)


Done. Created YOLO dataset at: DATA/YOLO_FISH_SEG


In [ ]:
# # First, ensure the correct library version is installed
# # If you haven't installed it yet, run: !pip install ultralytics==8.3.0

# from ultralytics import YOLO

# # Load a pre-trained segmentation model
# model = YOLO('yolov8n-seg.pt')

# # Train the model with the specified arguments
# # All the command-line arguments are passed as keyword arguments (kwargs)
# results = model.train(
#     data="fish_seg.yaml",
#     imgsz=640,
#     epochs=100,
#     batch=8,
#     hsv_h=0.015,
#     hsv_s=0.7,
#     hsv_v=0.4,
#     degrees=10,
#     translate=0.1,
#     scale=0.5,
#     shear=5,
#     device='0'  # Use 'cpu' or '0' for GPU if available
# )

In [3]:
from ultralytics import YOLO

# Load the trained segmentation model
model_path = r"runs/segment/train/weights/best.pt"
model = YOLO(model_path)

# Run prediction
source_path = r"data/test"
results = model.predict(
    source=source_path,
    imgsz=640,
    save=True,         # Equivalent to save=True in the CLI
    save_txt=False,    # Equivalent to save_txt=False in the CLI
    conf=0.25,
    # device=0         # Optional: Add this if you want to explicitly use your GPU
)


image 1/6 d:\Projects\MATSYA\Codes\data\test\00006.png: 512x640 1 fish, 179.1ms
image 2/6 d:\Projects\MATSYA\Codes\data\test\00078.png: 512x640 1 fish, 83.5ms
image 3/6 d:\Projects\MATSYA\Codes\data\test\00109.png: 512x640 1 fish, 77.5ms
image 4/6 d:\Projects\MATSYA\Codes\data\test\00210.png: 512x640 1 fish, 73.5ms
image 5/6 d:\Projects\MATSYA\Codes\data\test\00274.png: 512x640 1 fish, 81.5ms
image 6/6 d:\Projects\MATSYA\Codes\data\test\00668.png: 512x640 1 fish, 75.5ms
Speed: 4.7ms preprocess, 95.1ms inference, 4.7ms postprocess per image at shape (1, 3, 512, 640)
Results saved to runs\segment\predict


Build species-crop dataset from your Indian + Pakistan images

In [4]:
# D:\Projects\MATSYA\codes\make_species_crops.py
import os, glob, shutil, math
from pathlib import Path
import numpy as np
import cv2
from ultralytics import YOLO

# --------- CONFIG ----------
SEG_MODEL = r"runs/segment/train/weights/best.pt"

INDIAN_ROOT = r"data/Indian Seafood Fish Dataset"   # each subfolder = species
PAK_ROOT    = r"data/pak fishes dataset"  # each subfolder = species

OUT_CROPS   = r"data/species_crops"
CONF_THRESH = 0.25
PAD_RATIO   = 0.12   # 12% padding around bbox
MIN_AREA    = 2000   # ignore tiny detections (px^2)
# ---------------------------

def all_images(root):
    exts = (".jpg",".jpeg",".png",".bmp",".tif",".tiff")
    for sp_dir in sorted(Path(root).glob("*")):
        if not sp_dir.is_dir(): continue
        label = sp_dir.name.strip()
        for img in sp_dir.rglob("*"):
            if img.suffix.lower() in exts:
                yield str(img), label

def ensure(p): os.makedirs(p, exist_ok=True)

def poly_to_mask(poly, h, w):
    mask = np.zeros((h,w), dtype=np.uint8)
    cv2.fillPoly(mask, [np.array(poly, dtype=np.int32)], 255)
    return mask

def bbox_from_poly(poly, h, w, pad_ratio=0.1):
    xy = np.array(poly, dtype=np.float32)
    x1,y1 = xy.min(0)
    x2,y2 = xy.max(0)
    cx, cy = (x1+x2)/2, (y1+y2)/2
    bw, bh = (x2-x1), (y2-y1)
    padw, padh = bw*pad_ratio, bh*pad_ratio
    x1, y1 = max(0, int(cx - (bw/2 + padw))), max(0, int(cy - (bh/2 + padh)))
    x2, y2 = min(w-1, int(cx + (bw/2 + padw))), min(h-1, int(cy + (bh/2 + padh)))
    return x1,y1,x2,y2

def run(folder):
    model = YOLO(SEG_MODEL)
    ensure(OUT_CROPS)
    items = list(all_images(folder))
    for idx, (img_path, label) in enumerate(items, 1):
        img = cv2.imread(img_path)
        if img is None: continue
        h, w = img.shape[:2]
        # predict
        res = model.predict(source=img, verbose=False, conf=CONF_THRESH, imgsz=640)[0]
        if res.masks is None: 
            continue
        # each mask polygon -> crop
        for j, poly in enumerate(res.masks.xy):
            # poly comes as Nx2 (float pixel coords)
            if len(poly) < 6: 
                continue
            area = cv2.contourArea(np.array(poly, dtype=np.float32))
            if area < MIN_AREA: 
                continue
            x1,y1,x2,y2 = bbox_from_poly(poly, h, w, PAD_RATIO)
            crop = img[y1:y2, x1:x2].copy()
            if crop.size == 0: 
                continue
            out_dir = os.path.join(OUT_CROPS, label)
            ensure(out_dir)
            base = Path(img_path).stem
            out_path = os.path.join(out_dir, f"{base}_{j}.jpg")
            cv2.imwrite(out_path, crop)
        if idx % 200 == 0:
            print(f"[{idx}/{len(items)}] processed")

# Run both datasets
if os.path.exists(INDIAN_ROOT):
    run(INDIAN_ROOT)
if os.path.exists(PAK_ROOT):
    run(PAK_ROOT)

print("Done. Crops written to:", OUT_CROPS)


[200/9007] processed
[400/9007] processed
[600/9007] processed
[800/9007] processed
[1000/9007] processed
[1200/9007] processed
[1400/9007] processed
[1600/9007] processed
[1800/9007] processed
[2000/9007] processed
[2200/9007] processed
[2400/9007] processed
[2600/9007] processed
[2800/9007] processed
[3000/9007] processed
[3200/9007] processed
[3400/9007] processed
[3600/9007] processed
[3800/9007] processed
[4000/9007] processed
[4200/9007] processed
[4600/9007] processed
[4800/9007] processed
[5000/9007] processed
[5200/9007] processed
[5400/9007] processed
[5600/9007] processed
[5800/9007] processed
[6000/9007] processed
[6200/9007] processed
[6400/9007] processed
[6800/9007] processed
[7000/9007] processed
[7600/9007] processed
[200/753] processed
[400/753] processed
[600/753] processed
Done. Crops written to: data/species_crops
